# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy pandas matplotlib --upgrade pip --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import platform
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import subprocess
import re
import xml.etree.ElementTree as ET
import sys

**Declarations**

Set the variables below and run the generation cell. 

- `scenario` is the path to the `.xml` scenario file that will be simulated.
- `legi_nr` will be put as seed for the `.xml` scenario.

In [ ]:
# --- Declarations: editable variables for students ---
scenario = ...
legi_nr = ...

### 1. Run a Scenario

#### 1.1. Update Local Repository

In [ ]:
# --- Jemula Class Paths ---
# jemula and jemula802 are one repository, cloned into PROJECT_DIR below.
# Every path here is built from it, so moving the checkout means editing one
# line.
PROJECT_DIR = "jemula-project"
REPO_URL = "https://gitlab.ethz.ch/wireless/jemula.git"

lib = f"{PROJECT_DIR}/jemula/lib"
common = f"./:{PROJECT_DIR}/jemula802/bin:{PROJECT_DIR}/jemula/bin"
classpath_jemula = f"{common}:{lib}/xstream-1.3.1.jar:{lib}/jfreechart-1.5.3.jar:{lib}/jcommon-1.0.23.jar"
classpath_jemula802 = f"{classpath_jemula}:{lib}/commons-io-2.14.0.jar"

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")

In [ ]:
# --- Clone or Update the Jemula Repository ---
# One repository holding both jemula and jemula802, with the tests, the
# scenarios and the fixes.
if os.path.isdir(f"{PROJECT_DIR}/.git"):
    print(f"Updating {PROJECT_DIR} ...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=True)
    print(f"{PROJECT_DIR} updated.")
else:
    print(f"Cloning {PROJECT_DIR} ...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    print(f"{PROJECT_DIR} cloned.")


#### 1.2 Update Scenario Seed

In [ ]:
# --- Update Scenario Seed from Matriculation Number (preserve formatting) ---
def update_seed_in_scenario_text(scenario_path, new_seed):
    with open(scenario_path, "r", encoding="utf-8") as f:
        content = f.read()

    # replace seed="..." only inside <JE802StatEval ...>
    new_content = re.sub(
        r'(<JE802StatEval[^>]*\bseed=")(\d+)(")',
        rf'\g<1>{new_seed}\3',
        content,
        count=1
    )

    if content != new_content:
        with open(scenario_path, "w", encoding="utf-8") as f:
            f.write(new_content)
        print(f"Updated seed to {new_seed} in {scenario_path}")
    else:
        print("No seed attribute found.")

update_seed_in_scenario_text(scenario, legi_nr)

#### 1.3. Start the Scenario

In [ ]:
print(f"Running example scenario {scenario} ...")

process = subprocess.Popen(
    ["java", "-Xms2G", "-Xmx2G", "-classpath", classpath_jemula802, "emulator.JE802Starter", scenario],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

process.wait()
print(f"\nExample scenario finished with exit code {process.returncode}")

### 2 Simulation Results

#### 2.1. Folder Location
We dynamically extract the folder location from the property `Path2Results` to see where the files are saved to.

In [ ]:
# --- Locate Results Folder ---
tree = ET.parse(scenario)
root = tree.getroot()
path_elem = root.find(".//JE802StatEval")
path2results = path_elem.get("Path2Results") if path_elem is not None else None

if not path2results:
    raise ValueError("Path2Results not found in scenario XML")

print(f"Found results folder: {path2results}")

# List contents (cross-platform)
if os.path.isdir(path2results):
    print(f"\nContents of {path2results}:")
    for f in os.listdir(path2results)[:10]:
        print("  - ", f)
    print("  -  ...")
else:
    print(f"\nResults folder {path2results} does not exist yet.")